In [1]:
def _get_prefixes():
    prefixes = [
        'PREFIX onner: <http://purl.org/spatialai/onner/onner-full#>',
        'PREFIX data: <http://purl.org/spatialai/onner/onner-full/data#>',
        'PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>',
        'PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>',
        'PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>',
        'PREFIX owl: <http://www.w3.org/2002/07/owl#>',
    ]

    return '\n'.join(prefixes) + '\n'

In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON

# DATA RETRIEVAL FROM GRAPHDB
def _get_train_data(repo):

    try:
        # specify the repository
        sparql = SPARQLWrapper(f'{repo}')

        # query => retrieving data
        query = f"""
            {_get_prefixes()}

            SELECT ?paragraph ?entity ?offset ?length ?label ?status
            WHERE {{
                ?entId rdf:type onner:LabeledTerm ;
                       onner:labeledTermText ?entity ;
                       onner:offset ?offset ;
                       onner:length ?length ;
                       onner:labeledTermDirectlyContainedBy ?paraId ;
                       onner:hasLabeledTermStatus ?status .

                ?paraId onner:paragraphText ?paragraph .

                ?status onner:statusAssignmentDate ?datetime ;
                        onner:hasLabeledTermLabel ?labelId .

                ?labelId onner:labelText ?label .

                FILTER NOT EXISTS {{
                    ?entId onner:hasLabeledTermStatus ?newerStatus .
                    ?newerStatus onner:statusAssignmentDate ?newerDatetime .

                    FILTER(?newerDatetime > ?datetime)
                }}

                FILTER(
                    !STRSTARTS(STR(?status), STR(data:Rejected_)) &&
                    !STRSTARTS(STR(?status), STR(data:Candidate_))
                )
            }}
            ORDER BY ?paraId ?offset
        """

        # set query
        sparql.setQuery(query)

        # convert results to JSON
        sparql.setReturnFormat(JSON)

        # execute query
        results = sparql.query().convert()

        return results

    except Exception as e:
        print(f'Error querying the SPARQL endpoint: {e}')
        return None

In [3]:
def generate_train_data_spacy(repo):

    results = _get_train_data(repo)
    records = results['results']['bindings']
    length = len(records)
    start_index = 0
    index_ranges = []
    
    for i in range(length-1):
        if records[i]['paragraph']['value'] != records[i+1]['paragraph']['value']:
            end_index = i + 1
            index_ranges.append([start_index, end_index])
            start_index = end_index
    
    index_ranges.append([start_index, len(records)])
    
    annotations = []
    
    for range_ in index_ranges:
        start_index = range_[0]
        end_index = range_[1]
        
        paragraph = records[start_index]['paragraph']['value']
        entities = []
        for j in records[start_index:end_index]:
            entity = j['entity']['value']
            offset = j['offset']['value']
            length = j['length']['value']
            label = j['label']['value']
            status = j['status']['value']
    
            start_span = int(offset)
            end_span = start_span + int(length)
            entities.append([start_span, end_span, label])
    
        annotations.append([paragraph, {'entities': entities}])
    
    spacy_data = {
        'classes': [
            'CHEM_ENT',
            'MAT_ENT_STRUCT',
            'MAT_ENT_UNSTRUCT',
            'PROPERTY',
            'END_USE',
            "PROCESS",
            'EQUIPMENT',
            'MEASUREMENT',
            'ABBREVIATION'
        ],
        'annotations': annotations
    }
    
    return spacy_data


In [4]:
import json
from datetime import datetime

def main():
    
    graphdb_repo = 'http://dev:7200/repositories/Demon_FOIS' 
    spacy_data = generate_train_data_spacy(graphdb_repo)
    timestamp = datetime.now().strftime('%y%m%d%H%M%S')
    
    with open(f'training_data_spacy_{timestamp}.json', 'w', encoding='utf-8') as f:
        json.dump(spacy_data, f, indent=4, ensure_ascii=False)

In [5]:
if __name__ == '__main__':
    main()